In [1]:
# === ARC_ATLAS v3: long, low-LR run focused on fine details ===
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, subprocess, shlex
import tensorflow as tf
from tensorflow.keras import mixed_precision

# ---------- Paths ----------
CUDA_ID       = "0"
RUN_ROOT      = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3")
TRAIN_DIR     = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")
TRAIN_T1      = TRAIN_DIR / "t1"       # use separated folders to avoid duplicates
TRAIN_MASKS   = TRAIN_DIR / "masks"
MODULE_PATH   = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")

# ---------- New run folders (won't overwrite old) ----------
RUN_ID       = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR      = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR    = RUN_DIR / "models"
CALLBACKS_DIR= RUN_DIR / "callbacks"
LOG_DIR      = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

# ---------- Env & precision ----------
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session(); gc.collect()
mixed_precision.set_global_policy("mixed_float16")

# ---------- Tee logs to both notebook and file ----------
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for s in self.streams: s.write(data); s.flush()
        return len(data)
    def flush(self):
        for s in self.streams: s.flush()
log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("set_memory_growth failed:", e)

# ---------- Import training module; avoid MirroredStrategy to save VRAM ----------
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf  # in case the module references tf early
spec.loader.exec_module(seg)
seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

# ---------- Hyperparameters (fine detail bias, long schedule) ----------
INPUT_SHAPE    = (192, 224, 192, 1)
BATCH_SIZE     = 1
BASE_FILTERS   = 6      # match the architecture that trained well
SAM_HEADS      = 2
VAL_SPLIT      = 0.15
TOTAL_EPOCHS   = 300
INITIAL_EPOCH  = 0

# Learning-rate schedule knobs (lower LR, slight warmup)
INITIAL_LR     = 1.5e-5
MIN_LR         = 5e-7
WARMUP_EPOCHS  = 5

# Augmentation & small-lesion emphasis (these are accepted by your config)
AUG_INTENSITY          = 0.45
ROTATION_RANGE         = 25
SMALL_LESION_THRESHOLD = 6000
SYNTHETIC_LESION_PROB  = 0.60

# Loss mixture (tilt toward boundaries for thin/tiny lesions)
DICE_WEIGHT     = 0.30
BOUNDARY_WEIGHT = 0.70

# Optional regularization (only if your config supports these)
DROPOUT_RATE = 0.60
L2_REG       = 1.0e-3

# ---------- Launch training: absolutely fresh (no resume/no load) ----------
try:
    history = seg.train_dynamic_model(
        # Data roots – pass separated subfolders to avoid duplicate pairs
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,

        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,

        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,

        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,

        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
        SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
        ROTATION_RANGE=ROTATION_RANGE,
        VALIDATION_SPLIT=VAL_SPLIT,

        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,

        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run dir:", RUN_DIR)

except Exception as e:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(2):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try: log_file.flush()
    except Exception: pass


2025-11-07 15:24:52.894775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20251107_152454
TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Strategy: _DefaultDistributionStrategy


2025-11-07 15:24:54,893 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-07 15:24:54,893 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-07 15:24:54,893 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 1
2025-11-07 15:24:54,896 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2025-11-07 15:24:54,896 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1762554295.001229 1354291 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762554295.002235 1354291 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2025-11-07 15:24:55,005 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.73GB | GPU mem track

2025-11-07 15:25:37,511 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      1,194 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      4,050 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 6)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/300


...

2025-11-07 21:58:18,991 - SmartSOTA_Dynamic - INFO - Memory at epoch_299_end: CPU=11.00GB | GPU mem tracking failed | Disk: 1230.5GB free


Epoch 300: dice=0.1354 val_dice=0.2906 loss=0.3427 val_loss=0.2963 lr=5.00e-07
258/258 ━━━━━━━━━━━━━━━━━━━━ 76s 293ms/step - dice_coefficient: 0.1354 - loss: 0.3427 - val_dice_coefficient: 0.2906 - val_loss: 0.2963 - learning_rate: 5.0000e-07


2025-11-07 21:58:19,502 - SmartSOTA_Dynamic - INFO - 💾 Saved final weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454/models/smart_sota_dynamic_20251107_152454.final.weights.h5


2025-11-07 21:58:20,018 - SmartSOTA_Dynamic - INFO - 💾 Saved full model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454/models/smart_sota_dynamic_20251107_152454.keras
2025-11-07 21:58:20,018 - SmartSOTA_Dynamic - INFO - 🏁 Training complete.


Training complete. Logged keys: ['dice_coefficient', 'loss', 'val_dice_coefficient', 'val_loss', 'learning_rate']
Run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454
Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454
[CSV] last val_dice: 0.290649 (epoch 299)
[CSV] best  val_dice: 0.293619 (epoch 240)
Source file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454/callbacks/history.csv


In [1]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")


Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454
[CSV] last val_dice: 0.290649 (epoch 299)
[CSV] best  val_dice: 0.293619 (epoch 240)
Source file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251107_152454/callbacks/history.csv


# Train v3.1 resume v3 training

In [1]:
# === ARC_ATLAS_Train_v3 — rollback-to-v2-ish settings (fresh start) ==========
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, shlex, subprocess
import tensorflow as tf
from tensorflow.keras import mixed_precision

# --------- Paths ----------
CUDA_ID = "0"
TRAIN_ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global")
TRAIN_DIR   = TRAIN_ROOT / "train_hires"
TRAIN_T1    = TRAIN_DIR / "t1"      # <- use separated subfolder ONLY
TRAIN_MASKS = TRAIN_DIR / "masks"   # <- use separated subfolder ONLY

RUN_ROOT   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3")
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")

# --------- New run folders ----------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
LOG_DIR = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

# --------- Env & TF init ----------
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session(); gc.collect()
mixed_precision.set_global_policy("mixed_float16")

# Optional: tee logs to file and console
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data): 
        for s in self.streams: s.write(data); s.flush()
        return len(data)
    def flush(self): 
        for s in self.streams: s.flush()
log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("set_memory_growth failed:", e)

# --------- Import training module; avoid MirroredStrategy on 1 GPU ----------
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf
spec.loader.exec_module(seg)

# Force default (no mirrored). Saves VRAM and matches earlier good runs.
seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

# --------- Hyperparams (v2-ish) ----------
INPUT_SHAPE   = (192, 224, 192, 1)
BATCH_SIZE    = 1
BASE_FILTERS  = 8
SAM_HEADS     = 2
AUG_INTENSITY = 0.30
VAL_SPLIT     = 0.15
TOTAL_EPOCHS  = 140
INITIAL_EPOCH = 0

# LR schedule (the one that worked)
INITIAL_LR   = 1e-4
MIN_LR       = 5e-7
WARMUP_EPOCHS= 15

# --------- Launch training (FRESH: no resume, no load) ----------
try:
    history = seg.train_dynamic_model(
        # data roots: pass the main dir + explicit subfolders to avoid duplicates
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,

        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,

        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,

        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,

        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        VALIDATION_SPLIT=VAL_SPLIT,

        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run artifacts at:", RUN_DIR)

except Exception as e:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(3):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try: log_file.flush()
    except Exception: pass


2025-11-10 10:18:11.797154: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Run ID: 20251110_101813
TF: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Strategy: _DefaultDistributionStrategy


2025-11-10 10:18:13,687 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-10 10:18:13,687 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-10 10:18:13,688 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 1
2025-11-10 10:18:13,690 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (192, 224, 192, 1)
2025-11-10 10:18:13,690 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
I0000 00:00:1762795093.791410 1531329 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762795093.792425 1531329 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
2025-11-10 10:18:13,795 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.73GB | GPU mem track

2025-11-10 10:18:57,043 - SmartSOTA_Dynamic - INFO - Model: "SmartSOTA_Dynamic"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 192, 224,  │          0 │ -                 │
│ (InputLayer)        │ 192, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_conv_block │ (None, 192, 224,  │      2,024 │ input_layer[0][0] │
│ (ResidualConvBlock) │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vision_mamba_block  │ (None, 192, 224,  │      7,192 │ residual_conv_bl… │
│ (VisionMambaBlock)  │ 192, 8)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

Epoch 1/140


...

2025-11-10 13:11:18,277 - SmartSOTA_Dynamic - INFO - Memory at epoch_139_end: CPU=11.19GB | GPU mem tracking failed | Disk: 1230.3GB free


Epoch 140: dice=0.5935 val_dice=0.6694 loss=0.1713 val_loss=0.1411 lr=5.00e-07
258/258 ━━━━━━━━━━━━━━━━━━━━ 75s 292ms/step - dice_coefficient: 0.5935 - loss: 0.1713 - val_dice_coefficient: 0.6694 - val_loss: 0.1411 - learning_rate: 5.0000e-07


2025-11-10 13:11:18,813 - SmartSOTA_Dynamic - INFO - 💾 Saved final weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/models/smart_sota_dynamic_20251110_101813.final.weights.h5


2025-11-10 13:11:19,376 - SmartSOTA_Dynamic - INFO - 💾 Saved full model to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/models/smart_sota_dynamic_20251110_101813.keras
2025-11-10 13:11:19,376 - SmartSOTA_Dynamic - INFO - 🏁 Training complete.


Training complete. Logged keys: ['dice_coefficient', 'loss', 'val_dice_coefficient', 'val_loss', 'learning_rate']
Run artifacts at: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813
Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813
[CSV] last val_dice: 0.669352 (epoch 139)
[CSV] best  val_dice: 0.669352 (epoch 139)
Source file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/callbacks/history.csv


In [1]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")


Using RUN_DIR: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813
[CSV] last val_dice: 0.669352 (epoch 139)
[CSV] best  val_dice: 0.669352 (epoch 139)
Source file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20251110_101813/callbacks/history.csv
